# Hybrydowa analiza wielospektralna: Integracja algebry liniowej i uczenia maszynowego w ekstrakcji cech litologicznych

**Wstęp i Sformułowanie Problemu Badawczego**

Klasyfikacja pikseli na podstawie wielospektralnych zobrazowań satelitarnych (np. Sentinel-2) wymaga rozwiązania problemu wysokiej współliniowości rejestrowanych pasm spektralnych oraz zjawiska piksela mieszanego (mixed pixel). Zjawisko to występuje na krawędziach formacji geomorfologicznych, takich jak odsłonięcia skalne czy wyrobiska, gdzie sygnatura spektralna stanowi kombinację liniową albedo skały i otaczającego tła.

W niniejszym eksperymencie zaproponowano hybrydowy potok analityczny. Redukcja wymiarowości i ekstrakcja cech metodami algebry liniowej (Rozkład SVD, Spectral Angle Mapper) poprzedza nienadzorowaną segmentację optymalizacyjną algorytmem K-Means. Procedura ta eliminuje konieczność stosowania arbitralnych progów odcięcia, automatyzując proces decyzyjny w nowo zdefiniowanej przestrzeni cech.

In [ ]:
# Importy bibliotek analitycznych i przestrzennych
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Parametry globalne silnika graficznego
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titlesize'] = 12
# 1. Opcjonalnie: reset do domyślnego, jasnego stylu matplotlib
plt.style.use('default')

# 2. Globalne wymuszenie białego tła i czarnych napisów
plt.rcParams.update({
    "figure.facecolor": "white",       # Białe tło całej figury (w notatniku)
    "axes.facecolor": "white",         # Białe tło samego obszaru roboczego
    "savefig.facecolor": "white",      # Białe tło przy zapisie do pliku (np. png)
    "savefig.transparent": False,      # Wyłączenie przezroczystości przy zapisie
    "text.color": "black",             # Czarny kolor tekstów (np. tytułów)
    "axes.labelcolor": "black",        # Czarny kolor etykiet osi
    "xtick.color": "black",            # Czarny kolor podziałki X
    "ytick.color": "black"             # Czarny kolor podziałki Y
})
# ==============================================================================
# KONFIGURACJA PRZESTRZENI BADAWCZEJ
# ==============================================================================

# Zdefiniuj, który zestaw danych analizujesz: 'almeria', 'belchatow', 'andaluzja'
NAZWA_LOKACJI = 'almeria'

# System dynamicznie zbuduje sciezke (np. ./data/almeria/)
FOLDER_DANYCH = os.path.join(".", "data", NAZWA_LOKACJI)

# Slowniki punktow referencyjnych przypisane do konkretnych lokacji
# Wymagaja uzupelnienia o poprawne wspolrzedne dla danej mapy
PUNKTY_REFERENCYJNE = {
    'almeria': {
        "Srodek_Mapy_Almeria": (-2.40601, 36.87168), # Wklej tu wynik z diagnostyki
    }
}

# Walidacja obecnosci lokacji
if NAZWA_LOKACJI not in PUNKTY_REFERENCYJNE:
    raise KeyError(f"Brak zdefiniowanych punktow dla lokacji: {NAZWA_LOKACJI}")

PUNKTY_AKTYWNE = PUNKTY_REFERENCYJNE[NAZWA_LOKACJI]

PASMA_10M = ["B02", "B03", "B04", "B08"]
PASMA_20M = ["B05", "B06", "B07", "B8A", "B11", "B12"]

def pobierz_sciezki_danych(katalog, lista_pasm):
    sciezki = []
    for pasmo in lista_pasm:
        # Precyzyjne przeszukiwanie tylko w podkatalogu docelowym
        wzorzec = os.path.join(katalog, "**", f"*{pasmo}*.*")
        znalezione = [p for p in glob.glob(wzorzec, recursive=True) if p.lower().endswith(('.tif', '.tiff', '.jp2'))]
        if not znalezione:
            raise FileNotFoundError(f"Brak pliku dla pasma: {pasmo} w {katalog}")
        sciezki.append(znalezione[0])
    return sciezki

# Inicjalizacja strumieni danych
sciezki_10m = pobierz_sciezki_danych(FOLDER_DANYCH, PASMA_10M)
sciezki_20m = pobierz_sciezki_danych(FOLDER_DANYCH, PASMA_20M)

print("=========================================================================")
print(f"SRODOWISKO ZAINICJALIZOWANE: Projekt '{NAZWA_LOKACJI.upper()}'")
print(f"Katalog bazowy: {os.path.abspath(FOLDER_DANYCH)}")
print(f"Dostepne punkty kontrolne: {list(PUNKTY_AKTYWNE.keys())}")
print("=========================================================================")

## 1. Akwizycja i Geoprzetwarzanie Obserwacji Satelitarnych

Zobrazowania Sentinel-2 charakteryzują się heterogeniczną rozdzielczością przestrzenną. Pasma widzialne i bliskiej podczerwieni (VNIR) operują na siatce 10 m, podczas gdy pasma RedEdge oraz podczerwieni krótkofalowej (SWIR) na siatce 20 m.

W celu umożliwienia operacji macierzowych na pełnym spektrum, implementowana jest interpolacja dwuliniowa (bilinear resampling), reprojektująca kanały 20-metrowe do referencyjnej siatki 10-metrowej. Powstała trójwymiarowa kostka danych jest następnie spłaszczana do formatu analitycznego $\mathcal{M} \in \mathbb{R}^{N \times C}$, gdzie $N$ to liczba pikseli, a $C = 10$ to liczba pasm.

In [ ]:
def przygotuj_tensor_obserwacji(sciezki_10, sciezki_20):
    # Wczytywanie parametrow przestrzennych i ujednolicanie siatki
    pasma = []
    with rasterio.open(sciezki_10[0]) as src:
        h, w = src.height, src.width
        crs_zrodlowy = src.crs

    for s in sciezki_10:
        with rasterio.open(s) as src:
            pasma.append(src.read(1))

    for s in sciezki_20:
        with rasterio.open(s) as src:
            pasma.append(src.read(1, out_shape=(h, w), resampling=Resampling.bilinear))

    # Zlozenie list do struktury tensorowej i splaszczenie do macierzy 2D
    macierz_3d = np.dstack(pasma)
    macierz_2d = macierz_3d.reshape(h * w, macierz_3d.shape[2])
    return macierz_3d, macierz_2d, (h, w), crs_zrodlowy


def normalizacja_radiometryczna_rgb(macierz_3d):
    # Adaptacyjne skalowanie histogramu na bazie percentyli (2 i 98)
    rgb_out = np.zeros((macierz_3d.shape[0], macierz_3d.shape[1], 3), dtype=np.float32)

    # Mapowanie kanalow Sentinel-2: B04 (Indeks 2), B03 (Indeks 1), B02 (Indeks 0)
    indeksy_rgb = [2, 1, 0]

    for i, idx in enumerate(indeksy_rgb):
        kanal = macierz_3d[:, :, idx].astype(np.float32)
        # Ignorowanie wartosci tła (nodata) przy statystyce
        maska = kanal > 0

        if np.any(maska):
            p_min, p_max = np.percentile(kanal[maska], (2, 98))
            kanal_norm = (kanal - p_min) / (p_max - p_min + 1e-8)
            rgb_out[:, :, i] = np.clip(kanal_norm, 0, 1)

    return rgb_out


# Egzekucja operacji I/O
m_3d, m_2d, ksztalt_mat, crs_bazy = przygotuj_tensor_obserwacji(sciezki_10m, sciezki_20m)

# Ekstrakcja znormalizowanej kompozycji RGB
rgb = normalizacja_radiometryczna_rgb(m_3d)

plt.imshow(rgb)
plt.title("Kompozycja Rzeczywista R-G-B (Adaptacyjne Wyrownywanie Histogramu)", color = "black")
plt.axis('off')
plt.savefig("RGB_Sentinel_almeria.png", bbox_inches='tight', dpi=300, facecolor='white', transparent=False)
plt.show()

## 2. Aproksymacja Niskorandowa (SVD) i Analiza Głównych Składowych (PCA)

Macierz obserwacji $X \in \mathbb{R}^{N \times 10}$ wykazuje znaczną współliniowość. W celu wyizolowania sygnału o najwyższej wariancji (np. sygnatur litologicznych kopalni) od szumu radiometrycznego, implementuje się dekompozycję SVD na scentrowanej i wystandaryzowanej macierzy. Faktoryzacja macierzy przyjmuje postać:
$$X = U \Sigma V^T$$

**Zarządzanie anomaliami brzegowymi:** Zobrazowania satelitarne po przetworzeniu do mniejszych wycinków często zawierają obszary braku danych (NoData). Bezpośrednie zastosowanie algorytmu PCA na takiej macierzy skutkuje silnym zaburzeniem wariancji przez tło (powstanie anomalnych, skrajnych wartości na krawędziach przestrzeni zredukowanej). Wdrożono zatem mechanizm maskowania, który na etapie estymacji macierzy kowariancji i wektorów własnych wyklucza z obliczeń piksele o wartościach zerowych.

In [ ]:
def faktoryzacja_svd_potegowa(X, dim=3, iter_max=100):
    # Estymacja na bazie zdeflowanej macierzy kowariancji
    C = np.dot(X.T, X)
    cechy = X.shape[1]
    wektory_v = []
    wartosci_sigma = []

    C_robocza = C.copy()
    for k in range(dim):
        v = np.random.rand(cechy)
        v /= np.linalg.norm(v)

        for _ in range(iter_max):
            v_n = np.dot(C_robocza, v)
            norm = np.linalg.norm(v_n)
            if norm < 1e-9: break
            v = v_n / norm

        lam = np.dot(v.T, np.dot(C_robocza, v))
        sig = np.sqrt(lam)

        wektory_v.append(v)
        wartosci_sigma.append(sig)

        # Deflacja macierzy
        C_robocza -= lam * np.outer(v, v)

    return np.array(wartosci_sigma), np.array(wektory_v).T

def transformacja_pca_robust(dane_2d, ksztalt_3d, dim=3):
    # Krok 1: Definicja maski dla pikseli posiadajacych informacje
    maska_aktywnosci = np.sum(dane_2d, axis=1) > 0
    dane_fizyczne = dane_2d[maska_aktywnosci]

    # Krok 2: Statystyka obliczana WYLACZNIE na prawdziwych danych
    mu = np.mean(dane_fizyczne, axis=0)
    std = np.std(dane_fizyczne, axis=0)

    # Krok 3: Standaryzacja izometryczna (Z-score)
    X_std = (dane_fizyczne - mu) / (std + 1e-8)

    # Krok 4: Dekompozycja SVD przestrzeni odszumionej
    Sigma, Vt = faktoryzacja_svd_potegowa(X_std, dim=dim)
    pca_aktywne = np.dot(X_std, Vt)

    # Krok 5: Rekonstrukcja topologii macierzy rastrowej
    pca_kompletne = np.zeros((dane_2d.shape[0], dim))
    pca_kompletne[maska_aktywnosci] = pca_aktywne

    return pca_kompletne.reshape(ksztalt_3d[0], ksztalt_3d[1], -1), Sigma, maska_aktywnosci

# Egzekucja transformacji ortogonalnej
pca_3d, wektor_sigma, maska_tla = transformacja_pca_robust(m_2d, ksztalt_mat, dim=3)
pca_2d = pca_3d.reshape(ksztalt_mat[0] * ksztalt_mat[1], -1)

# Wizualizacja odseparowanych struktur przestrzennych
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
tytuly = ['PC1 (Dominujaca Wariancja)', 'PC2', 'PC3']

for i in range(3):
    kanal_pc = pca_3d[:, :, i]
    aktywne_dane = kanal_pc[maska_tla.reshape(ksztalt_mat)]

    # Skalowanie wylacznie na podstawie waznych pikseli
    if np.any(aktywne_dane):
        p_min, p_max = np.percentile(aktywne_dane, (2, 98))
    else:
        p_min, p_max = -1, 1

    img = ax[i].imshow(kanal_pc, cmap='gray', vmin=p_min, vmax=p_max)
    ax[i].set_title(tytuly[i], color = "black")
    ax[i].axis('off')
    fig.colorbar(img, ax=ax[i], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("PC1-3_almeria_podglad.png", bbox_inches='tight', dpi=300, facecolor='white', transparent=False)
plt.show()

## 3. Diagnostyka Kompresji Spektralnej i Interpretacja Optyczna PCA

W celu walidacji stopnia kompresji informacji wykorzystywany jest **współczynnik wyjaśnionej wariancji** (Explained Variance Ratio - EVR). Wartości własne macierzy kowariancji $\lambda_i$, będące kwadratami wartości osobliwych $\sigma_i^2$, określają wkład $i$-tej składowej w całkowitą wariancję systemu. Graficzna interpretacja tego zjawiska nosi nazwę **Wykresu Osypiska** (Scree Plot).

Dodatkowo, zredukowana przestrzeń wektorowa ulega rekonstrukcji w formie kompozycji w fałszywych barwach (False Color Composite). Rzutowanie składowych PC1, PC2 i PC3 odpowiednio na kanały RGB ekranu pozwala na dramatyczne wyostrzenie granic litologicznych i wydzielenie różnych jednostek stratygraficznych (np. podział osadów na dnie wyrobiska), które w domenie pasma widzialnego (R-G-B) pozostają niewidoczne dla ludzkiego oka z powodu silnej autokorelacji sygnału.

In [ ]:
def statystyka_rozkladu_eigen(wektor_osobliwy):
    # Relacja wartosci osobliwych do wartosci wlasnych macierzy kowariancji
    wariancje = wektor_osobliwy ** 2
    var_total = np.sum(wariancje)
    return (wariancje / var_total) * 100

def skalowanie_kanalu_pc(kanal, p_min=2, p_max=98):
    # Odporna na obserwacje skrajne normalizacja afiniczna
    maska = kanal != 0  # Opieramy sie wylacznie na aktywnych przestrzeniach
    if not np.any(maska): return np.zeros_like(kanal)

    val_min, val_max = np.percentile(kanal[maska], (p_min, p_max))
    kanal_out = (kanal - val_min) / (val_max - val_min + 1e-8)
    return np.clip(kanal_out, 0, 1)

# Estymacja udzialu wariancji
procenty_wariancji = statystyka_rozkladu_eigen(wektor_sigma)

# Architektura wykresow diagnostycznych
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

# Wykres Osypiska
slupki = ax[0].bar([f'PC{i+1}' for i in range(len(procenty_wariancji))], procenty_wariancji, color='#2c3e50')
for s in slupki:
    h = s.get_height()
    ax[0].text(s.get_x() + s.get_width()/2., h + 0.5, f'{h:.2f}%', ha='center', va='bottom', fontweight='bold')

ax[0].set_title("Wykres Osypiska (Explained Variance Ratio)")
ax[0].set_ylabel("Udzial Wariancji Informacyjnej [%]")
ax[0].grid(axis='y', linestyle='--', alpha=0.7)

# Rekonstrukcja False Color z niezaleznych wektorow PCA
# R = PC1 (glowna wariancja), G = PC2 (ortogonalne kontrasty), B = PC3 (subtelne niuanse)
pca_rgb = np.dstack((
    skalowanie_kanalu_pc(pca_3d[:, :, 0]),
    skalowanie_kanalu_pc(pca_3d[:, :, 1]),
    skalowanie_kanalu_pc(pca_3d[:, :, 2])
))

ax[1].imshow(pca_rgb)
ax[1].set_title("Sygnatura Zredukowana w Falszywych Kolorach\n(R: PC1, G: PC2, B: PC3)", color = "black")
ax[1].axis('off')

plt.tight_layout()
plt.savefig("PC1-3_almeria_wykres_falsecolor.png", bbox_inches='tight', dpi=300,facecolor='white', transparent=False)

plt.show()

## 4. Analiza Wektorów Ładunków (PCA Loadings) – Interpretacja Fizyczna

Dekompozycja SVD umożliwia nie tylko kompresję danych, ale również śledzenie proweniencji sygnału. Wykorzystując prawostronne wektory osobliwe (macierz $V^T$), konstruujemy **macierz ładunków PCA (PCA Loadings)**. Wektor ładunku określa korelację liniową (wagę) pomiędzy oryginalnymi kanałami spektralnymi instrumentu MSI a nowo wyznaczonymi składowymi ortogonalnymi.

Pozwala to na fizyczną dekonstrukcję wariancji:
* Dominacja konkretnego pasma w pierwszej składowej ($PC_1$) wskazuje na wektor albedo (całkowitą jasność radiometryczną obiektów, różnicę między ciemnym lasem a jasnym osypiskiem).
* Aktywność pasm wegetacyjnych (NIR/RedEdge) lub geologicznych (SWIR) w $PC_2$ lub $PC_3$ sugeruje wyodrębnienie specyficznego indeksu różnicowego (np. uwydatnienie zawartości tlenków żelaza na tle uogólnionego rozkładu skał).

In [ ]:
def ekstrakcja_ladunkow_pca(dane_2d, maska, dim=4):
    """
    Ekstrakcja wag dla składowych ortogonalnych w celu interpretacji
    wplywu pasm zrodlowych na proces kompresji SVD.
    """
    dane_aktywne = dane_2d[maska]
    mu = np.mean(dane_aktywne, axis=0)
    std = np.std(dane_aktywne, axis=0)
    X_std = (dane_aktywne - mu) / (std + 1e-8)

    # Wykorzystujemy wczesniej zdefiniowana funkcje potegowa dla wiekszej liczby wymiarow
    _, Vt_matrix = faktoryzacja_svd_potegowa(X_std, dim=dim)

    return Vt_matrix

# Odtwarzamy przestrzen nazw pasm uzytych w eksperymencie (kolejnosc ze stosu 3D)
nazwy_kanalow = PASMA_10M + PASMA_20M

# Ekstrakcja macierzy korelacji miedzy oryginalnymi pasmami a przestrzenia PC
macierz_ladunkow = ekstrakcja_ladunkow_pca(m_2d, maska_tla.reshape(-1), dim=4)

# Wizualizacja cieplna struktury wag wektorowych
fig, ax = plt.subplots(figsize=(12, 6))

im_ladunki = ax.imshow(macierz_ladunkow.T, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(nazwy_kanalow)))
ax.set_yticks(np.arange(macierz_ladunkow.shape[1]))
ax.set_xticklabels(nazwy_kanalow, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels([f'PC{i+1}' for i in range(macierz_ladunkow.shape[1])], fontsize=10, fontweight='bold')

ax.set_title("Struktura Ładunków PCA (Wpływ Kanałów MSI na Zredukowane Wymiary)", fontsize=12, pad=15)
ax.set_xlabel("Oryginalne Kanały Sentinel-2", labelpad=10)
ax.set_ylabel("Składowe Ortogonalne", labelpad=10)

fig.colorbar(im_ladunki, ax=ax, fraction=0.02, pad=0.04, label="Współczynnik Korelacji (Ładunek)")

for i in range(macierz_ladunkow.shape[1]):
    for j in range(len(nazwy_kanalow)):
        ax.text(j, i, f"{macierz_ladunkow.T[i, j]:.2f}",
                ha="center", va="center", color="black" if -0.5 < macierz_ladunkow.T[i, j] < 0.5 else "white",
                fontsize=9)

plt.tight_layout()
plt.savefig("PC1-4_alemria_wplyw.png", bbox_inches='tight', dpi=300, facecolor='white', transparent=False)

plt.show()

## 4a. Inżynieria Cech: Eliminacja Wektora Albedo i Analiza Głębokich Składowych

Analiza wektorów ładunków (Loadings) wykazała, że pierwsza składowa główna ($PC_1$), niosąca niemal 90% wariancji układu, jest silnie skorelowana ze wszystkimi pasmami w sposób dodatni. W fizyce teledetekcji oznacza to, że $PC_1$ reprezentuje całkowite albedo obiektu oraz efekty topograficzne (np. zacienienie stoków wyrobiska).

Aby wyizolować czysty sygnał litologiczny i mineralogiczny, stosuje się technikę odrzucenia dominującej składowej. Rozszerzono dekompozycję przestrzeni do czterech wymiarów ($k=4$), a następnie zrekonstruowano kompozycję fałszywych kolorów z pominięciem $PC_1$. Rzutowanie składowych $PC_2$, $PC_3$ i $PC_4$ odpowiednio na kanały R, G, B pozwala na spłaszczenie optyczne rzeźby terenu i radykalne wyostrzenie chemicznych granic między formacjami skalnymi, wegetacją i osadami.

In [ ]:
# Ekstrakcja 4-wymiarowej podprzestrzeni SVD
pca_4d_3d, wektor_sigma_4d, maska_tla_4d = transformacja_pca_robust(m_2d, ksztalt_mat, dim=4)

# Rekonstrukcja False Color z pominieciem wektora jasnosci (PC1)
# R = PC2 (Wegetacja/Grunt), G = PC3 (Litologia A), B = PC4 (Litologia B)
pca_bez_albedo = np.dstack((
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 1]),  # Indeks 1 to PC2
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 2]),  # Indeks 2 to PC3
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 3])   # Indeks 3 to PC4
))

# Wizualizacja porownawcza
fig, ax = plt.subplots(1, 2, figsize=(18, 8))

# Przypomnienie klasycznego PCA (z dominacja cieni i albedo)
pca_rgb_klasyczne = np.dstack((
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 0]),
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 1]),
    skalowanie_kanalu_pc(pca_4d_3d[:, :, 2])
))

ax[0].imshow(pca_rgb_klasyczne)
ax[0].set_title("Przestrzen Pełna (PC1, PC2, PC3)\nZauwazalny silny wplyw jasnosci i cieni")
ax[0].axis('off')

# Kompozycja bez PC1
ax[1].imshow(pca_bez_albedo)
ax[1].set_title("Przestrzen Wyizolowana (PC2, PC3, PC4)\nEliminacja albedo wyostrza struktury chemiczne/mineralne")
ax[1].axis('off')

plt.tight_layout()
plt.savefig("PC1-4_almeria_porownanie1v4.png", bbox_inches='tight', dpi=300)

plt.show()

# Aktualizacja wektorow wejsciowych dla kolejnych krokow (SAM i K-Means)
# Przekazujemy wylacznie zredukowana przestrzen bez PC1
pca_do_analizy_3d = pca_4d_3d[:, :, 1:4]
pca_do_analizy_2d = pca_do_analizy_3d.reshape(ksztalt_mat[0] * ksztalt_mat[1], 3)

## 5. Klasyfikacja Wektorowa: Wielokryterialna Analiza Angularna (Spectral Angle Mapper)

W klasycznej analizie spektralnej algorytm SAM służy do wyznaczania podobieństwa fizykochemicznego obiektów poprzez pomiar odchylenia angularnego (kąta $\theta$) między wektorem badanego piksela $v$ a wielowymiarowym wektorem referencyjnym $r$ (wzorcem litologicznym). Podstawową zaletą tej metryki jest jej niezmienność względem skalowania długości wektora, co czyni ją odporną na zmiany natężenia oświetlenia wywołane topografią terenu.

W celu walidacji założeń teoretycznych, w tej sekcji zaimplementowano eksperyment porównawczy. Algorytm wyznacza dystrybucję kątową w dwóch odrębnych podprzestrzeniach cech geometrycznych:
1. W przestrzeni składowych pierwszego rzędu ($PC_1, PC_2, PC_3$), gdzie wariancja jest zdominowana przez globalną energię albedo.
2. W przestrzeni cech głębokich ($PC_2, PC_3, PC_4$), z której w kroku inżynierii cech całkowicie wyeliminowano składową jasnościową.

Porównanie wynikowych matryc angularnych pozwoli na ocenę, w jakim stopniu obecność wektora albedo ($PC_1$) modyfikuje kierunki wektorów spektralnych i wpływa na selektywność klasyfikatora w strefach cieniowania topograficznego. Aby zapewnić elastyczność potoku, system pobiera współrzędne z dynamicznego słownika zdefiniowanego na początku notatnika.

In [ ]:
# Wybor obiektu testowego ze slownika aktywnej lokacji
# Upewnij sie, ze punkt nalezy do 'almeria' np: "Szklarnie_Centrum"
NAZWA_OBIEKTU = list(PUNKTY_AKTYWNE.keys())[0] # Pobiera pierwszy zdefiniowany
WSPOLRZEDNE = PUNKTY_AKTYWNE[NAZWA_OBIEKTU]

def transformacja_sam_silnik(mat_3d, wektor_ref):
    h, w, c = mat_3d.shape
    M = mat_3d.reshape(h * w, c)

    iloczyn = np.dot(M, wektor_ref)
    norma_m = np.linalg.norm(M, axis=1)
    norma_w = np.linalg.norm(wektor_ref)

    cos_val = np.clip(iloczyn / (norma_m * norma_w + 1e-8), -1.0, 1.0)
    theta = np.arccos(cos_val)
    return theta.reshape(h, w)

# DIAGNOSTYKA SCIEZKI DOSTĘPU: Wyswietlenie pliku uzytego do georeferencji
sciezka_bezwzgledna = os.path.abspath(sciezki_10m[0])
print("=========================================================================")
print(f"CEL BADANIA: {NAZWA_OBIEKTU} (Lon: {WSPOLRZEDNE[0]}, Lat: {WSPOLRZEDNE[1]})")
print(f"Plik georeferencji: {sciezka_bezwzgledna}")
print("=========================================================================")

# Reprojekcja geodezyjna punktu kontrolnego
with rasterio.open(sciezki_10m[0]) as rds:
    x_p, y_p = transform('EPSG:4326', rds.crs, [WSPOLRZEDNE[0]], [WSPOLRZEDNE[1]])
    row, col = rds.index(x_p[0], y_p[0])

    print(f"Geometria obrazu wejsciowego: {rds.width}x{rds.height} pikseli")
    print(f"Indeksy macierzy: Row (Y) = {row}, Col (X) = {col}")

    if not (0 <= row < rds.height and 0 <= col < rds.width):
        bnd = rds.bounds
        raise ValueError(
            f"Punkt {NAZWA_OBIEKTU} lezy poza kadrem sceny.\n"
            f"Krawedzie mapy (UTM): X[{bnd.left:.0f}, {bnd.right:.0f}], Y[{bnd.bottom:.0f}, {bnd.top:.0f}]"
        )

# Obliczenie Wariantu A: Pelna przestrzen PCA (PC1 + PC2 + PC3)
wzorzec_wariant_A = pca_4d_3d[row, col, :3]
mapa_sam_wariant_A = transformacja_sam_silnik(pca_4d_3d[:, :, :3], wzorzec_wariant_A)

# Obliczenie Wariantu B: Przestrzen zredukowana, wyizolowana (PC2 + PC3 + PC4)
wzorzec_wariant_B = pca_do_analizy_3d[row, col, :]
mapa_sam_wariant_B = transformacja_sam_silnik(pca_do_analizy_3d, wzorzec_wariant_B)

# Zestawienie komparatywne wynikowych macierzy odleglosci
fig, ax = plt.subplots(1, 2, figsize=(18, 8))

imA = ax[0].imshow(mapa_sam_wariant_A, cmap='magma_r')
ax[0].set_title(f"Wariant A: Kat SAM w przestrzeni PC1-3")
ax[0].axis('off')
fig.colorbar(imA, ax=ax[0], fraction=0.046, pad=0.04, label="Odleglosc angularna [rad]")

imB = ax[1].imshow(mapa_sam_wariant_B, cmap='magma_r')
ax[1].set_title(f"Wariant B: Kat SAM w przestrzeni PC2-4")
ax[1].axis('off')
fig.colorbar(imB, ax=ax[1], fraction=0.046, pad=0.04, label="Odleglosc angularna [rad]")

plt.tight_layout()
plt.savefig("SAM_almeria.png", bbox_inches='tight', dpi=300)

plt.show()

## 6. Synergia Algebry i Uczenia Maszynowego: Optymalizacja Granic (K-Means)

Zastosowanie ciągłej mapy odległości kątowej SAM, nawet w optymalizowanej przestrzeni pozbawionej wektora albedo, niesie ryzyko subiektywizmu przy ręcznym definiowaniu progów odcięcia dla pikseli mieszanych.

Aby w pełni zautomatyzować proces decyzyjny, wdrażany jest model uczenia nienadzorowanego. Konstruowana jest finalna macierz obserwacji $F$, zawierająca wektory głębokiej przestrzeni strukturalnej ($PC_2, PC_3, PC_4$) oraz metrykę litologiczną ($\theta_{SAM}$ z Wariantu B).

Na przestrzeni tej (po uprzedniej normalizacji izometrycznej wariancji), iteruje algorytm **K-Means**. Poprzez minimalizację kryterium inercji (wariancji wewnątrzklastrowej), model automatycznie wyznacza nieliniowe hiperpłaszczyzny podziału (tesselacja Voronoia), precyzyjnie izolując zdefiniowane struktury przestrzenne. Ostateczny wynik podlega georeferencji i serializacji do formatu GeoTIFF.

In [ ]:
def klastrowanie_hybrydowe_ml(pca_macierz_2d, sam_macierz_2d, K=4):
    # Fuzja cech fizycznych i metryki angularnej
    sam_1d = sam_macierz_2d.flatten().reshape(-1, 1)
    cechy_zlaczone = np.hstack((pca_macierz_2d, sam_1d))

    # Izometria przestrzeni w celu zachowania normy Euklidesowej w algorytmie grupujacym
    skaler = StandardScaler()
    cechy_znormalizowane = skaler.fit_transform(cechy_zlaczone)

    print(f"-> Uruchamianie optymalizatora K-Means (K={K})...")
    klasyfikator = KMeans(n_clusters=K, random_state=42, n_init=10)
    etykiety_1d = klasyfikator.fit_predict(cechy_znormalizowane)

    return etykiety_1d.reshape(sam_macierz_2d.shape)

def eksport_danych_gis(sciezka_bazy, sciezka_wyjscia, macierz_wynikowa):
    # Przepisanie metadanych afinicznych z pliku referencyjnego
    with rasterio.open(sciezka_bazy) as baza:
        meta = baza.profile

    meta.update(
        dtype=rasterio.uint8,
        count=1,
        compress='lzw',
        nodata=None
    )

    with rasterio.open(sciezka_wyjscia, 'w', **meta) as cel:
        cel.write(macierz_wynikowa.astype(rasterio.uint8), 1)

# Egzekucja modelu uzywajac optymalnego Wariantu B (odpornego na albedo)
klastry_ml_2d = klastrowanie_hybrydowe_ml(pca_do_analizy_2d, mapa_sam_wariant_B, K=6)

# Zestawienie wynikow
fig, ax = plt.subplots(1, 2, figsize=(18, 8))

ax[0].imshow(rgb)
ax[0].set_title(f"Sygnal Optyczny R-G-B ({NAZWA_LOKACJI.upper()})")
ax[0].axis('off')

im_klastry = ax[1].imshow(klastry_ml_2d, cmap='tab10')
ax[1].set_title(f"K-Means dla PC1 - PC3")
ax[1].axis('off')
fig.colorbar(im_klastry, ax=ax[1], fraction=0.046, pad=0.04, label="Sygnatura Klastra")

plt.tight_layout()
plt.savefig("kmeans_vs_rgb.png", bbox_inches='tight', dpi=300)

plt.show()

# Utworzenie pliku wynikowego dla systemow GIS (np. QGIS, ArcGIS)
nazwa_pliku_out = f"Segmentacja_ML_{NAZWA_LOKACJI.upper()}.tif"
eksport_danych_gis(sciezki_10m[0], nazwa_pliku_out, klastry_ml_2d)
print(f"=========================================================================")
print(f"ZAKONCZONO SUKCESEM: Wygenerowano mape klasyfikacji {nazwa_pliku_out}")
print(f"=========================================================================")

## 7. Paradoks Niezmienniczości Skali i Zaawansowana Fuzja Cech (Model Albedo)

Wynik klasyfikacji z poprzedniego kroku uwidacznia fundamentalny problem fizyczny znany w teledetekcji jako paradoks niezmienniczości skali (Scale Invariance). Algorytm Kąta Spektralnego (SAM) estymuje wyłącznie odległość angularną, ignorując całkowicie moduł (długość) wektora widmowego. W efekcie obiekty o skrajnie różnym albedo – takie jak tafla wody (silna absorpcja, wektor o minimalnej amplitudzie) oraz infrastruktura szklarniowa (silne odbicie kierunkowe, wektor o maksymalnej amplitudzie) – charakteryzują się wysoką współliniowością. Odległość kątowa $\theta$ między nimi dąży do zera, co skutkuje ich błędną agregacją do pojedynczego klastra.

Usunięcie składowej $PC_1$ w poprzednich krokach (w celu eliminacji cieni topograficznych) pozbawiło model jedynego wymiaru zdolnego do liniowej separacji tych klas.

Aby rozwiązać ten problem, wdrażana jest zaawansowana fuzja cech. Algorytm K-Means operuje na rozszerzonej, 5-wymiarowej przestrzeni obserwacji. Model otrzymuje pełną macierz głównych składowych ($PC_1, PC_2, PC_3, PC_4$) zintegrowaną z mapą kątową $\theta_{SAM}$ wyznaczoną w przestrzeni odszumionej (Wariant B). Dzięki temu system zachowuje odporność na anomalię oświetlenia w terenie górskim, odzyskując jednocześnie zdolność dyskryminacji materiałów na podstawie ich globalnej wariancji energetycznej ($PC_1$). Zwiększono również parametr $K=5$, aby poprawnie zamodelować wyższą wariancję zaktualizowanej przestrzeni.

In [ ]:
def klastrowanie_hybrydowe_zaawansowane(pca_4d_mat, sam_macierz_2d, K=5):
    # Fuzja pelnej przestrzeni ortogonalnej (z PC1) z metryka angularna (bez PC1)
    h, w, c = pca_4d_mat.shape
    pca_plaskie = pca_4d_mat.reshape(h * w, c)

    sam_1d = sam_macierz_2d.flatten().reshape(-1, 1)
    cechy_zlaczone = np.hstack((pca_plaskie, sam_1d))

    # Standaryzacja wymiarow w celu zapewnienia izometrii dystansu Euklidesowego
    skaler = StandardScaler()
    cechy_znormalizowane = skaler.fit_transform(cechy_zlaczone)

    klasyfikator = KMeans(n_clusters=K, random_state=42, n_init=10)
    etykiety_1d = klasyfikator.fit_predict(cechy_znormalizowane)

    return etykiety_1d.reshape(h, w)

# Egzekucja zaktualizowanego modelu ML z uwzglednieniem wektora jasnosci (PC1)
klastry_zaawansowane_2d = klastrowanie_hybrydowe_zaawansowane(pca_4d_3d, mapa_sam_wariant_B, K = 6)

# Zestawienie komparatywne ewolucji modelu ML
fig, ax = plt.subplots(1, 2, figsize=(18, 8))

# Prezentacja bledu algorytmu bazowego (z Komorki 16)
im_baza = ax[0].imshow(klastry_ml_2d, cmap='tab10')
ax[0].set_title("Model Bazowy (Przestrzen pozbawiona PC1)\nWidoczna bledna agregacja wody i szklarni")
ax[0].axis('off')
fig.colorbar(im_baza, ax=ax[0], fraction=0.046, pad=0.04, ticks=range(4), label="Indeks Klastra")

# Prezentacja rozwiazania zaawansowanego
im_zaaw = ax[1].imshow(klastry_zaawansowane_2d, cmap='tab10')
ax[1].set_title(f"Model Zaawansowany (Fuzja PC1-PC4 + SAM Wariant B)\nBezbłędna dyskryminacja obiektów (K=6)")
ax[1].axis('off')
fig.colorbar(im_zaaw, ax=ax[1], fraction=0.046, pad=0.04, ticks=range(5), label="Indeks Klastra")

plt.tight_layout()
plt.savefig("kmeans_comp.png", bbox_inches='tight', dpi=300)

plt.show()

# Serializacja optymalnego wyniku do formatu przestrzennego
nazwa_pliku_out_zaawansowana = f"Segmentacja_ML_ZAAWANSOWANA_{NAZWA_LOKACJI.upper()}.tif"
eksport_danych_gis(sciezki_10m[0], nazwa_pliku_out_zaawansowana, klastry_zaawansowane_2d)
print(f"=========================================================================")
print(f"ZAKONCZONO SUKCESEM: Zapisano skorygowana mape przestrzenna -> {nazwa_pliku_out_zaawansowana}")
print(f"=========================================================================")

## 8. Analiza Komparatywna (Ablation Study) i Weryfikacja Heurystyczna

W celu oceny jakości i wydajności autorskiego potoku analitycznego (Fuzja PCA + SAM), przeprowadzono bezpośrednie zestawienie wyników z modelem bazowym (Baseline). Model bazowy zdefiniowano jako algorytm K-Means aplikowany w sposób naiwny na 10-wymiarową przestrzeń surowych pasm wielospektralnych, pozbawioną kompresji oraz normalizacji efektów albedo.

Ewaluacja obejmuje kryterium:
1. **Jakościowe (Heurystyczne):** Odporność modelu na cienie topograficzne w strefach rzeźby górskiej (szum albedo) oraz zdolność do separacji klas silnie kontrastujących geometrycznie (morze vs. infrastruktura sztuczna).
2. **Obliczeniowe:** Redukcja czasu zbieżności algorytmu wynikająca z kompresji wielowymiarowej macierzy danych ($d=10 \rightarrow d=5$).

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# ==============================================================================
# MODEL BAZOWY (NAIWNY) - SUROWE 10 PASM
# ==============================================================================
skaler_surowy = StandardScaler()
m_2d_znormalizowane = skaler_surowy.fit_transform(m_2d)

print("-> Trenowanie modelu naiwnego (10 wymiarów)...")
kmeans_naiwny = KMeans(n_clusters=6, random_state=42, n_init=10)
klastry_naiwne_1d = kmeans_naiwny.fit_predict(m_2d_znormalizowane)
klastry_naiwne_2d = klastry_naiwne_1d.reshape(ksztalt_mat)

# ==============================================================================
# MODEL HYBRYDOWY (ZAAWANSOWANY) - 5 WYMIARÓW (PC1-4 + SAM)
# ==============================================================================
# (Wykorzystujemy przestrzen zoptymalizowana w Komorce 18)
h, w, c = pca_4d_3d.shape
pca_plaskie = pca_4d_3d.reshape(h * w, c)
sam_1d = mapa_sam_wariant_B.flatten().reshape(-1, 1)
cechy_zlaczone = np.hstack((pca_plaskie, sam_1d))

skaler_zaawansowany = StandardScaler()
cechy_znormalizowane = skaler_zaawansowany.fit_transform(cechy_zlaczone)

print("-> Trenowanie modelu hybrydowego (5 wymiarów z fuzją albedo)...")
kmeans_hybrydowy = KMeans(n_clusters=6, random_state=42, n_init=10)
klastry_hybrydy_1d = kmeans_hybrydowy.fit_predict(cechy_znormalizowane)
klastry_zaawansowane_2d = klastry_hybrydy_1d.reshape(h, w)

# ==============================================================================
# WIZUALIZACJA I RAPORT WYNIKÓW
# ==============================================================================
fig, ax = plt.subplots(1, 3, figsize=(24, 8))

# 1. Rzeczywistość RGB
ax[0].imshow(rgb)
ax[0].set_title("Optyczny Obraz Referencyjny (R-G-B)")
ax[0].axis('off')

# 2. Model Bazowy (Naiwny)
im_naiwny = ax[1].imshow(klastry_naiwne_2d, cmap='Set1')
ax[1].set_title("Model Bazowy (Surowe 10 Pasm)")
ax[1].axis('off')
fig.colorbar(im_naiwny, ax=ax[1], fraction=0.046, pad=0.04, label="Indeks Klastra")

# 3. Model Zoptymalizowany (Hybryda)
im_hybryda = ax[2].imshow(klastry_zaawansowane_2d, cmap='Set1')
ax[2].set_title("Model Zoptymalizowany (Fuzja PCA + Kąt SAM)")
ax[2].axis('off')
fig.colorbar(im_hybryda, ax=ax[2], fraction=0.046, pad=0.04, label="Indeks Klastra")

plt.tight_layout()
plt.savefig("kmeans_all_vs_select.png", bbox_inches='tight', dpi=300)
plt.show()

## 9. Wzbogacenie Przestrzeni Cech (Feature Enrichment) i Analiza Redundancji

Zgodnie z wiodącą metodyką badawczą (m.in. w analizie pokrycia terenu w rejonie Morza Śródziemnego), skuteczność klasyfikatorów numerycznych często zależy od włączenia dedykowanych indeksów spektralnych, które transformują surowe odbicie radiometryczne w znormalizowane wskaźniki fizyko-chemiczne.

Do przestrzeni cech włączono:
1. **NDVI (Normalized Difference Vegetation Index):** Maksymalizuje separację aktywności fotosyntetycznej.
2. **NDWI (Normalized Difference Water Index):** Uwydatnia obecność zbiorników wodnych.
3. **NDBI (Normalized Difference Built-up Index):** Wskaźnik zurbanizowania / infrastruktury (wykorzystujący pasmo SWIR, silnie reagujący na beton, asfalt oraz plastiki szklarniowe).

**Analiza eksperymentalna (Ablation Study):**
Podczas prac badawczych przetestowano wpływ budowy 8-wymiarowej przestrzeni cech $[PC_{1-4}, \theta_{SAM}, NDVI, NDWI, NDBI]$ na skuteczność klasteryzacji.

In [ ]:
# ==============================================================================
# OBLICZANIE INDEKSÓW SPEKTRALNYCH
# ==============================================================================
# Ekstrakcja pasm z macierzy 3D (Zgodnie z indeksem: B03=1, B04=2, B08=3, B11=8)
b03 = m_3d[:, :, 1].astype(np.float32)
b04 = m_3d[:, :, 2].astype(np.float32)
b08 = m_3d[:, :, 3].astype(np.float32)
b11 = m_3d[:, :, 8].astype(np.float32)

# Obliczenia z zabezpieczeniem przed dzieleniem przez zero (1e-8)
ndvi = (b08 - b04) / (b08 + b04 + 1e-8)
ndwi = (b03 - b08) / (b03 + b08 + 1e-8)
ndbi = (b11 - b08) / (b11 + b08 + 1e-8)

# Splaszczenie macierzy do postaci kolumnowej (N pikseli x 1 cecha)
ndvi_1d = ndvi.flatten().reshape(-1, 1)
ndwi_1d = ndwi.flatten().reshape(-1, 1)
ndbi_1d = ndbi.flatten().reshape(-1, 1)

# ==============================================================================
# KONSTRUKCJA SUPER-PRZESTRZENI CECH (8 Wymiarów)
# ==============================================================================
# Bierzemy PC1, PC2, PC3, PC4 (z wczesniejszych komorek) oraz SAM
h, w, c = pca_4d_3d.shape
pca_plaskie = pca_4d_3d.reshape(h * w, c)
sam_1d = mapa_sam_wariant_B.flatten().reshape(-1, 1)

# Połączenie wszystkich cech w jedną macierz
cechy_super = np.hstack((pca_plaskie, sam_1d, ndvi_1d, ndwi_1d, ndbi_1d))

# Normalizacja izometryczna
skaler_super = StandardScaler()
X_super_znormalizowane = skaler_super.fit_transform(cechy_super)

print(f"-> Zbudowano 8-wymiarową przestrzeń cech o rozmiarze: {X_super_znormalizowane.shape}")
print("-> Trenowanie K-Means w 8-wymiarowej przestrzeni (PCA+SAM+Indeksy)...")

# 1. Trenowanie modelu na 8 wymiarach
kmeans_8d = KMeans(n_clusters=6, random_state=42, n_init=10)
klastry_8d_1d = kmeans_8d.fit_predict(X_super_znormalizowane)
klastry_8d_2d = klastry_8d_1d.reshape(h, w)

# 2. Wizualizacja porównawcza
fig, ax = plt.subplots(1, 2, figsize=(20, 8))

# Wynik 5D (Poprzedni najlepszy)
ax[0].imshow(klastry_zaawansowane_2d, cmap='Set1')
ax[0].set_title("Model 5D (PCA + SAM)")
ax[0].axis('off')

# Wynik 8D (Nowy z indeksami)
ax[1].imshow(klastry_8d_2d, cmap='Set1')
ax[1].set_title("Model 8D (PCA + SAM + NDVI/NDWI/NDBI)")
ax[1].axis('off')

plt.tight_layout()
plt.savefig("kmeans_plus.png", bbox_inches='tight', dpi=300)

plt.show()
# ==============================================================================
# WIZUALIZACJA INDEKSÓW SPEKTRALNYCH (DIAGNOSTYKA CECH)
# ==============================================================================
fig_ind, ax_ind = plt.subplots(1, 3, figsize=(22, 6))

# 1. NDVI (Wegetacja) - Paleta RdYlGn (czerwony dla niskiego, zielony dla wysokiego NDVI)
im_ndvi = ax_ind[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
ax_ind[0].set_title("NDVI - Wegetacja", fontweight='bold')
ax_ind[0].axis('off')
plt.colorbar(im_ndvi, ax=ax_ind[0], fraction=0.046, pad=0.04)

# 2. NDWI (Woda) - Paleta Blues
im_ndwi = ax_ind[1].imshow(ndwi, cmap='Blues', vmin=-0.3, vmax=0.5)
ax_ind[1].set_title("NDWI - Woda", fontweight='bold')
ax_ind[1].axis('off')
plt.colorbar(im_ndwi, ax=ax_ind[1], fraction=0.046, pad=0.04)

# 3. NDBI (Zabudowa/Szklarnie) - Paleta Magma (wysoki kontrast dla infrastruktury)
im_ndbi = ax_ind[2].imshow(ndbi, cmap='magma', vmin=-0.2, vmax=0.4)
ax_ind[2].set_title("NDBI - Infrastruktura", fontweight='bold')
ax_ind[2].axis('off')
plt.colorbar(im_ndbi, ax=ax_ind[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("spectral_indices_diagnostic.png", bbox_inches='tight', dpi=300)
plt.show()


Wprowadzenie dodatkowych wskaźników spektralnych (NDVI, NDWI, NDBI) do modelu klasyfikacyjnego zaowocowało zmianą priorytetów dyskryminacyjnych algorytmu. Zaobserwowano wyraźny kompromis między precyzją odwzorowania różnych typów pokrycia terenu:

* **Zysk szczegółowości (obszary roślinne):** Włączenie indeksów spektralnych, w szczególności NDVI, znacząco poprawiło separację terenów zielonych. Model zyskał zdolność do skuteczniejszego odróżniania różnych formacji roślinnych i typów biomasy, co wcześniej było utrudnione przez uśrednianie sygnatury w surowych danych.
* **Redukcja szczegółowości (obszary zurbanizowane):** Jednocześnie odnotowano spadek precyzji w odwzorowaniu detali miejskich. Zwiększona wymiarowość przestrzeni cech spowodowała, że skomplikowana struktura miejska (charakteryzująca się dużą zmiennością lokalną, np. cienie budynków, tekstura dachów, mieszane materiały) stała się bardziej homogeniczna w oczach algorytmu.

**Wniosek badawczy:** Wzbogacenie przestrzeni o wskaźniki spektralne jest wysoce efektywne w analizie środowisk naturalnych, jednak może prowadzić do nadmiernego wygładzenia ("uśrednienia") cech w złożonych środowiskach antropogenicznych. Wybór cech wejściowych powinien zatem wynikać z priorytetów badawczych – czy celem jest analiza biomasy, czy mapowanie infrastruktury miejskiej.